In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load all datasets and concatenate them with a dataset name column
ours = {
    'Avila': 'ovr_results/ours/Avila_results.csv',
    'Chessgame': 'ovr_results/ours/Chessgame_results.csv',
    'Covertype': 'ovr_results/ours/Covertype_results.csv',
    'Dermatology': 'ovr_results/ours/Dermatology_results.csv',
    'HAR': 'ovr_results/ours/HAR_results.csv',
    'Land-use': 'ovr_results/ours/Land-use_results.csv',
    # 'Mfeat_icdm21': 'ovr_results/ours/Mfeat_icdm21_results.csv',
    'Mfeat': 'ovr_results/ours/Mfeat_results.csv',
    # 'Mosquitoes': 'ovr_results/ours/Mosquitoes_results.csv',
    'Nursery': 'ovr_results/ours/Nursery_results.csv',
    'PhishingURL': 'ovr_results/ours/PhishingURL_results.csv',
    'Satimage': 'ovr_results/ours/Satimage_results.csv',
    'Walking': 'ovr_results/ours/Walking_results.csv',
}

kaggle = {
    'Cirrhosis': 'ovr_results/kaggle/cirrhosis_results.csv',
    'CustomerSegmentation': 'ovr_results/kaggle/customer_segmentation_results.csv',
    'FashionMNIST': 'ovr_results/kaggle/fashion-mnist_results.csv',
    'Healthcare': 'ovr_results/kaggle/healthcare_results.csv',
    'MusicGenre': 'ovr_results/kaggle/music_genre_results.csv',
    'PredictiveMaintenance': 'ovr_results/kaggle/predictive_maintenance_results.csv',
    'StarClassification': 'ovr_results/kaggle/star_classification_results.csv',
    'StudentPerformance': 'ovr_results/kaggle/student_performance_data_results.csv',
    'Zoo': 'ovr_results/kaggle/zoo_results.csv',
}

openml = {
    'Spectrometer': 'ovr_results/openml/dataset_313_spectrometer_results.csv',
    'AmazonReviews_1457': 'ovr_results/openml/dataset_1457_amazon-commerce-reviews_results.csv',
    'OneHundredPlants_Margin': 'ovr_results/openml/dataset_1491_one-hundred-plants-margin_results.csv',
    'OneHundredPlants_Shape': 'ovr_results/openml/dataset_1492_one-hundred-plants-shape_results.csv',
    'OneHundredPlants_Texture': 'ovr_results/openml/dataset_1493_one-hundred-plants-texture_results.csv',
    'BachChoralHarmony': 'ovr_results/openml/dataset_4552_BachChoralHarmony_results.csv',

    'AmazonReviews_seed0': 'ovr_results/openml/dataset_44478_amazon-commerce-reviews_seed_0_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    'AmazonReviews_seed1': 'ovr_results/openml/dataset_44479_amazon-commerce-reviews_seed_1_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    'AmazonReviews_seed2': 'ovr_results/openml/dataset_44480_amazon-commerce-reviews_seed_2_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    'AmazonReviews_seed3': 'ovr_results/openml/dataset_44481_amazon-commerce-reviews_seed_3_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    'AmazonReviews_seed4': 'ovr_results/openml/dataset_44482_amazon-commerce-reviews_seed_4_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',

    'FBI_Fingerprint': 'ovr_results/openml/fabert_results.csv',
    'FARS': 'ovr_results/openml/fars_results.csv',
    'Microaggregation2': 'ovr_results/openml/microaggregation2_results.csv',
}

quapy = {
    'Academic-Success': 'ovr_results/quapy/academic-success_results.csv',
    'Digits': 'ovr_results/quapy/digits_results.csv',
    'Dry-Bean': 'ovr_results/quapy/dry-bean_results.csv',
    'Letter': 'ovr_results/quapy/letter_results.csv',
    'Wine-Quality': 'ovr_results/quapy/wine-quality_results.csv',
}


uci = {
    'Wine': 'ovr_results/uci/wine_results.csv',
}

datasets = {**ours, **kaggle, **openml, **quapy, **uci}

# Load and concatenate all datasets
dfs = []
for dataset_name, filepath in datasets.items():
    temp_df = pd.read_csv(filepath)
    temp_df['dataset'] = dataset_name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)
# Drop columns ending with '_p'
df = df.drop(columns=[col for col in df.columns if col.endswith('_p')])

In [ ]:
len(df['dataset'].unique())

In [ ]:
datasets

In [ ]:
# df = pd.read_csv('ovr_results/Covertype_results.csv')
# df.head()

In [ ]:
import re

pattern = r'^c(.+?)_(?:p_normalized|real)$'

classes = sorted(set([
    re.match(pattern, col).group(1) 
    for col in df.columns 
    if re.match(pattern, col)
]))
classes

In [ ]:
import numpy as np

def normalized_cross_entropy(p, q, n_test=100):
    """
    p: true prevalence (array-like)
    q: estimated prevalence (array-like)
    n_test: test set size (int or array-like)

    returns: normCE(p, q)
    """
    eps = 0.5 / n_test

    p = np.clip(p, eps, 1 - eps)
    q = np.clip(q, eps, 1 - eps)

    ce_pq = -(p * np.log2(q) + (1 - p) * np.log2(1 - q))
    ce_pp = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

    return ce_pq - ce_pp

In [ ]:
# Calculate absolute error metrics
# Build all error columns at once to avoid fragmentation
error_cols = {}
for cls in classes:
    error_cols[f'c{cls}_error_normalized'] = abs(df[f'c{cls}_real'] - df[f'c{cls}_p_normalized'])

# Concatenate all new columns at once
df = pd.concat([df, pd.DataFrame(error_cols)], axis=1)

ce_cols = {}

for cls in classes:
    ce_cols[f'c{cls}_CEnorm'] = normalized_cross_entropy(
        p=df[f'c{cls}_real'].values,
        q=df[f'c{cls}_p_normalized'].values,
    )

# Concatenate all new columns at once
df = pd.concat([df, pd.DataFrame(ce_cols)], axis=1)

# Calculate MAE (Mean Absolute Error) for each row
# Sum all error columns and divide by number of non-NaN values
error_columns = [f'c{cls}_error_normalized' for cls in classes]
df['MAE'] = df[error_columns].mean(axis=1, skipna=True)

ce_columns = [f'c{cls}_CEnorm' for cls in classes]
df['mean_normCE'] = df[ce_columns].mean(axis=1, skipna=True)

In [ ]:
summarize_df = df[['qnt', 'MAE', 'mean_normCE', 'dataset']].copy()

In [ ]:
summarize_df['dataset'].unique()

# Analysis

## Individual

### Normalized vs Standard Error

In [ ]:
# def plot_error_distribution(df):
#     """
#     Create an interactive box plot showing error distribution by class and quantification method.
    
#     Parameters:
#     df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
#     Returns:
#     plotly.graph_objects.Figure: Interactive plotly figure
#     """
#     import plotly.graph_objects as go
    
#     # Get unique quantification methods
#     qnt_methods = df['qnt'].unique().tolist()
    
#     # Create figure
#     fig = go.Figure()
    
#     # Add traces for each qnt method
#     for qnt in qnt_methods:
#         df_filtered = df[df['qnt'] == qnt]
        
#         # Add normalized and non-normalized traces for each class
#         for i in classes:
#             # Normalized error
#             fig.add_trace(go.Box(
#                 y=df_filtered[f'c{i}_error_normalized'],
#                 name=f'Class {i} (Norm)',
#                 visible=(qnt == qnt_methods[0]),  # Only first method visible initially
#                 boxmean=True,
#                 marker=dict(color='lightblue'),
#                 legendgroup=f'class{i}',
#                 showlegend=True
#             ))
            
#             # Non-normalized error
#             fig.add_trace(go.Box(
#                 y=df_filtered[f'c{i}_error'],
#                 name=f'Class {i}',
#                 visible=(qnt == qnt_methods[0]),  # Only first method visible initially
#                 boxmean=True,
#                 marker=dict(color='lightcoral'),
#                 legendgroup=f'class{i}',
#                 showlegend=True
#             ))
    
#     # Create buttons for dropdown
#     buttons = []
    
#     for idx, qnt in enumerate(qnt_methods):
#         # Calculate which traces should be visible for this qnt method
#         visible = [False] * len(fig.data)
#         start_idx = idx * 12  # 12 traces per qnt method (6 classes × 2 error types)
#         for i in range(12):
#             visible[start_idx + i] = True
        
#         buttons.append(dict(
#             label=qnt,
#             method='update',
#             args=[{'visible': visible}]
#         ))
    
#     # Update layout
#     fig.update_layout(
#         updatemenus=[dict(
#             active=0,
#             buttons=buttons,
#             x=0.17,
#             xanchor='left',
#             y=1.15,
#             yanchor='top'
#         )],
#         title='Error Distribution by Class and Quantification Method',
#         xaxis_title='Class and Error Type',
#         yaxis_title='Error',
#         height=600,
#         showlegend=True
#     )
    
#     return fig

# # Call the function and display the plot
# fig = plot_error_distribution(df)
# fig.show()


### Boxplot per method

In [ ]:
def plot_normalized_error_by_method(df):
    """
    Create an interactive box plot showing normalized error distribution by class for each quantification method.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Create figure
    fig = go.Figure()
    
    # Get unique quantification methods
    qnt_methods = df['qnt'].unique().tolist()
    
    # Add traces for each qnt method
    for qnt in qnt_methods:
        df_filtered = df[df['qnt'] == qnt]
        
        # Add trace for each class
        for i in classes:
            fig.add_trace(go.Box(
                y=df_filtered[f'c{i}_error_normalized'],
                name=f'Class {i}',
                visible=(qnt == qnt_methods[0]),  # Only first method visible initially
                boxmean=True
            ))
    
    # Create buttons for dropdown
    buttons = []
    for idx, qnt in enumerate(qnt_methods):
        # Calculate which traces should be visible for this qnt method
        visible = [False] * len(fig.data)
        start_idx = idx * 6  # 6 traces per qnt method (one per class)
        for i in range(6):
            visible[start_idx + i] = True
        
        buttons.append(dict(
            label=qnt,
            method='update',
            args=[{'visible': visible}]
        ))
    
    # Update layout
    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title='Normalized Error Distribution by Class',
        xaxis_title='Class',
        yaxis_title='Normalized Error',
        height=600,
        showlegend=True
    )
    
    return fig

# Call the function and display the plot
fig2 = plot_normalized_error_by_method(df)
fig2.show()

In [ ]:
def plot_traditional_vs_syn_comparison(df):
    """
    Create an interactive box plot comparing traditional quantification methods with their synthetic counterparts.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Define method pairs (traditional, synthetic)
    method_pairs = {
        'ACC': ('ACC', 'ACC_syn'),
        'PACC': ('PACC', 'PACC_syn'),
        'X': ('X', 'X_syn'),
        'MAX': ('MAX', 'MAX_syn'),
        'T50': ('T50', 'T50_syn'),
        'MS': ('MS', 'MS_syn'),
        'MS2': ('MS2', 'MS2_syn'),
        'SMM': ('SMM', 'SMM_syn'),
        'HDy': ('HDy', 'HDy_syn'),
        'DyS': ('DyS', 'DySyn')
    }
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each method pair
    for method_name, (trad_method, syn_method) in method_pairs.items():
        # Filter data for traditional method
        df_trad = df[df['qnt'] == trad_method]
        df_syn = df[df['qnt'] == syn_method]
        
        # Add traces for each class
        for i in classes:
            # Traditional method
            if len(df_trad) > 0:
                fig.add_trace(go.Box(
                    y=df_trad[f'c{i}_error_normalized'],
                    name=f'Class {i} (Trad)',
                    visible=(method_name == 'ACC'),
                    boxmean=True,
                    marker=dict(color='lightcoral'),
                    legendgroup=f'class{i}',
                    showlegend=True,
                    offsetgroup=f'c{i}_trad',
                    width=0.3
                ))
            
            # Synthetic method
            if len(df_syn) > 0:
                fig.add_trace(go.Box(
                    y=df_syn[f'c{i}_error_normalized'],
                    name=f'Class {i} (Syn)',
                    visible=(method_name == 'ACC'),
                    boxmean=True,
                    marker=dict(color='lightblue'),
                    legendgroup=f'class{i}',
                    showlegend=True,
                    offsetgroup=f'c{i}_syn',
                    width=0.3
                ))
    
    # Create buttons for dropdown
    buttons = []
    for idx, method_name in enumerate(method_pairs.keys()):
        # Calculate which traces should be visible
        visible = [False] * len(fig.data)
        start_idx = idx * 12  # 12 traces per method pair (6 classes × 2 types)
        for i in range(12):
            visible[start_idx + i] = True
        
        buttons.append(dict(
            label=method_name,
            method='update',
            args=[{'visible': visible}]
        ))
    
    # Update layout
    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title='Traditional vs Synthetic Methods: Normalized Error by Class',
        xaxis_title='Class and Method Type',
        yaxis_title='Normalized Error',
        height=600,
        showlegend=True,
        boxmode='group'
    )
    
    return fig

# Call the function and display the plot
fig3 = plot_traditional_vs_syn_comparison(df)
fig3.show()


In [ ]:
df[df['qnt'] == 'DyS'].columns

### Win Plot

In [ ]:
def plot_win_comparison(df):
    """
    Create a horizontal bar plot comparing wins between traditional and synthetic methods.
    A method wins on a class if it has lower mean error than its counterpart.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Define method pairs (traditional, synthetic)
    method_pairs = {
        'ACC': ('ACC', 'ACC_syn'),
        'PACC': ('PACC', 'PACC_syn'),
        'X': ('X', 'X_syn'),
        'MAX': ('MAX', 'MAX_syn'),
        'T50': ('T50', 'T50_syn'),
        'MS': ('MS', 'MS_syn'),
        'MS2': ('MS2', 'MS2_syn'),
        'SMM': ('SMM', 'SMM_syn'),
        'HDy': ('HDy', 'HDy_syn'),
        'DyS': ('DyS', 'DySyn')
    }
    
    # Calculate wins for each method
    results = []
    
    for method_name, (trad_method, syn_method) in method_pairs.items():
        trad_wins = 0
        syn_wins = 0
        
        # Check each class
        for i in classes:
            df_trad = df[df['qnt'] == trad_method]
            df_syn = df[df['qnt'] == syn_method]
            
            if len(df_trad) > 0 and len(df_syn) > 0:
                trad_mean = df_trad[f'c{i}_error_normalized'].mean()
                syn_mean = df_syn[f'c{i}_error_normalized'].mean()
                
                if trad_mean < syn_mean:
                    trad_wins += 1
                elif syn_mean < trad_mean:
                    syn_wins += 1
        
        results.append({
            'method': method_name,
            'trad_wins': trad_wins,
            'syn_wins': syn_wins
        })
    
    # Create figure
    fig = go.Figure()
    
    # Extract data
    methods = [r['method'] for r in results]
    trad_wins = [-r['trad_wins'] for r in results]  # Negative for left side
    syn_wins = [r['syn_wins'] for r in results]
    
    # Add traditional wins (left side, negative values)
    fig.add_trace(go.Bar(
        y=methods,
        x=trad_wins,
        name='Traditional',
        orientation='h',
        marker=dict(color='lightcoral'),
        text=[-x for x in trad_wins],
        textposition='auto',
    ))
    
    # Add synthetic wins (right side, positive values)
    fig.add_trace(go.Bar(
        y=methods,
        x=syn_wins,
        name='Synthetic',
        orientation='h',
        marker=dict(color='lightblue'),
        text=syn_wins,
        textposition='auto',
    ))
    
    # Update layout
    fig.update_layout(
        title='Traditional vs Synthetic: Win Count by Method (6 classes)',
        xaxis_title='Number of Wins',
        yaxis_title='Method',
        barmode='relative',
        height=500,
        xaxis=dict(
            tickvals=[-6, -4, -2, 0, 2, 4, 6],
            ticktext=['6', '4', '2', '0', '2', '4', '6']
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return fig

# Call the function and display the plot
fig_wins = plot_win_comparison(df)
fig_wins.show()

## All results

In [ ]:
def plot_error_metrics_comparison(summarize_df):
    """
    Create an interactive box plot comparing error metrics (MAE and mean_normCE) 
    between traditional and synthetic methods.
    
    Parameters:
    summarize_df (pd.DataFrame): DataFrame containing qnt, MAE, mean_normCE, and dataset columns
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Get unique datasets
    datasets_list = summarize_df['dataset'].unique().tolist()
    
    # Define metrics
    metrics = ['MAE', 'mean_normCE']
    
    # Define method pairs (traditional, synthetic)
    method_pairs = [
        ('ACC', 'ACC_syn'),
        ('PACC', 'PACC_syn'),
        ('X', 'X_syn'),
        ('MAX', 'MAX_syn'),
        ('T50', 'T50_syn'),
        ('MS', 'MS_syn'),
        ('MS2', 'MS2_syn'),
        ('SMM', 'SMM_syn'),
        ('HDy', 'HDy_syn'),
        ('DyS', 'DySyn')
    ]
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each dataset, metric, and method pair combination
    for dataset in datasets_list:
        for metric in metrics:
            df_dataset = summarize_df[summarize_df['dataset'] == dataset]
            
            for trad_method, syn_method in method_pairs:
                # Traditional method
                df_trad = df_dataset[df_dataset['qnt'] == trad_method]
                if len(df_trad) > 0:
                    fig.add_trace(go.Box(
                        y=df_trad[metric],
                        name=f'{trad_method}',
                        visible=(dataset == datasets_list[0] and metric == metrics[0]),
                        boxmean=True,
                        marker=dict(color='lightcoral'),
                        legendgroup=trad_method,
                        showlegend=True,
                        offsetgroup=trad_method,
                        width=0.4,
                        meta={'dataset': dataset, 'metric': metric, 'method': trad_method, 'type': 'trad'}
                    ))
                
                # Synthetic method
                df_syn = df_dataset[df_dataset['qnt'] == syn_method]
                if len(df_syn) > 0:
                    fig.add_trace(go.Box(
                        y=df_syn[metric],
                        name=f'{syn_method}',
                        visible=(dataset == datasets_list[0] and metric == metrics[0]),
                        boxmean=True,
                        marker=dict(color='lightblue'),
                        legendgroup=syn_method,
                        showlegend=True,
                        offsetgroup=syn_method,
                        width=0.4,
                        meta={'dataset': dataset, 'metric': metric, 'method': syn_method, 'type': 'syn'}
                    ))
    
    # Create combined dropdown for Dataset - Metric
    combined_buttons = []
    for dataset in datasets_list:
        for metric in metrics:
            visible = []
            for trace in fig.data:
                visible.append(trace.meta['dataset'] == dataset and trace.meta['metric'] == metric)
            
            combined_buttons.append(dict(
                label=f'{dataset} - {metric}',
                method='update',
                args=[
                    {'visible': visible},
                    {'title': f'{metric} Distribution by Method - {dataset}',
                     'yaxis': {'title': metric}}
                ]
            ))
    
    # Update layout with combined dropdown
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=combined_buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.01,
                xanchor="left",
                y=1.15,
                yanchor="top"
            )
        ],
        annotations=[
            dict(text="Dataset - Metric:", showarrow=False,
                 x=0.01, y=1.18, xref="paper", yref="paper", align="left")
        ],
        title=f'{metrics[0]} Distribution by Method - {datasets_list[0]}',
        xaxis_title='Quantification Method',
        yaxis_title=metrics[0],
        height=600,
        showlegend=True,
        boxmode='group'
    )
    
    return fig

# Call the function and display the plot
fig_metrics = plot_error_metrics_comparison(summarize_df)
fig_metrics.show()

In [ ]:
summarize_df

In [ ]:
# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset['rank'] = ranking_by_dataset.groupby('dataset')['mean_normCE'].rank(method='min')
ranking_by_dataset = ranking_by_dataset.sort_values(['dataset', 'rank'])

# Pivot table to show rankings in a more readable format
ranking_pivot = ranking_by_dataset.pivot(index='qnt', columns='dataset', values='rank')
ranking_pivot = ranking_pivot.fillna('-')

print("Ranking of quantification methods by dataset (based on MAE):")
print("Lower rank = better performance (lower MAE)")
print("\n")
print(ranking_pivot)

# Also create a summary showing average rank across all datasets
avg_ranking = ranking_by_dataset.groupby('qnt')['rank'].mean().reset_index()
avg_ranking.columns = ['qnt', 'avg_rank']
avg_ranking = avg_ranking.sort_values('avg_rank')

# print("\n\nAverage ranking across all datasets:")
# print(avg_ranking)

In [ ]:
import plotly.graph_objects as go

# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset_mae = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset_mae['rank'] = ranking_by_dataset_mae.groupby('dataset')['mean_normCE'].rank(method='min')

# Create box plot of rankings per quantification method
fig = go.Figure()

# Get unique quantification methods sorted by median rank
qnt_methods = ranking_by_dataset_mae.groupby('qnt')['rank'].median().sort_values().index.tolist()

# Add box plot for each quantification method
for qnt in qnt_methods:
    qnt_data = ranking_by_dataset_mae[ranking_by_dataset_mae['qnt'] == qnt]
    fig.add_trace(go.Box(
        y=qnt_data['rank'],
        name=qnt,
        boxmean='sd',
        marker=dict(
            color='lightblue' if '_syn' in qnt or qnt in ['DySyn'] else 'lightcoral'
        )
    ))

# Update layout
fig.update_layout(
    title='Ranking Distribution of Quantification Methods Across Datasets (based on MAE)',
    xaxis_title='Quantification Method',
    yaxis_title='Rank (lower is better)',
    height=600,
    showlegend=False,
    # yaxis=dict(autorange='reversed')  # Lower rank at top
)

fig.show()

# Print summary statistics
print("Summary statistics of rankings across datasets:")
ranking_summary = ranking_by_dataset_mae.groupby('qnt')['rank'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
ranking_summary = ranking_summary.sort_values('mean')
print(ranking_summary)

In [ ]:
import plotly.graph_objects as go

# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset_mae = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset_mae['rank'] = ranking_by_dataset_mae.groupby('dataset')['mean_normCE'].rank(method='min')

# Create box plot of rankings per quantification method
fig = go.Figure()

# Get unique quantification methods sorted by median rank
qnt_methods = ranking_by_dataset_mae.groupby('qnt')['rank'].median().sort_values().index.tolist()

# Add box plot for each quantification method
for qnt in qnt_methods:
    qnt_data = ranking_by_dataset_mae[ranking_by_dataset_mae['qnt'] == qnt]
    fig.add_trace(go.Box(
        y=qnt_data['rank'],
        name=qnt,
        boxmean='sd',
        marker=dict(
            color='lightblue' if '_syn' in qnt or qnt in ['DySyn'] else 'lightcoral'
        )
    ))

# Update layout
fig.update_layout(
    title='Ranking Distribution of Quantification Methods Across Datasets (based on mean_normCE)',
    xaxis_title='Quantification Method',
    yaxis_title='Rank (lower is better)',
    height=600,
    showlegend=False,
    # yaxis=dict(autorange='reversed')  # Lower rank at top
)
fig.show()

# Print summary statistics
print("Summary statistics of rankings across datasets:")
ranking_summary = ranking_by_dataset_mae.groupby('qnt')['rank'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
ranking_summary = ranking_summary.sort_values('mean')
print(ranking_summary)